In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
    Run inference on a PyTorch model. Given an input tensor and a trained <code>torch.nn.Linear</code> model, compute the forward pass and store the result in the output tensor.
  </p>

  <p>
    The model performs a linear transformation: <code>output = input @ weight.T + bias</code> where <code>weight</code> has shape [output_size, input_size] and <code>bias</code> has shape [output_size].
  </p>

  <h2>Implementation Requirements</h2>
  <ul>
    <li>Use PyTorch's built-in functions and operations</li>
    <li>The <code>solve</code> function signature must remain unchanged</li>
    <li>The final result must be stored in the <code>output</code> tensor</li>
    <li>The model is already loaded and ready for inference</li>
  </ul>

  <h2>Example 1:</h2>
  <pre>
  Input:  input = [[1.0, 2.0]]  (batch_size=1, input_size=2)
          model: Linear layer with weight=[[0.5, 1.0], [1.5, 0.5]], bias=[0.1, 0.2]
  Output: output = [[2.6, 2.7]]  (batch_size=1, output_size=2)
  </pre>

  <h2>Example 2:</h2>
  <pre>
  Input:  input = [[1.0], [2.0], [3.0]]  (batch_size=3, input_size=1)
          model: Linear layer with weight=[[2.0]], bias=[1.0]
  Output: output = [[3.0], [5.0], [7.0]]  (batch_size=3, output_size=1)
  </pre>

  <h2>Constraints</h2>
  <ul>
    <li>1 ≤ <code>batch_size</code> ≤ 1,000</li>
    <li>1 ≤ <code>input_size</code> ≤ 1,000</li>
    <li>1 ≤ <code>output_size</code> ≤ 1,000</li>
    <li>-10.0 ≤ input values ≤ 10.0</li>

  <li>Performance is measured with <code>batch_size</code> = 1,000</li>
</ul>


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# input and model are on the GPU
def solve(input: jax.Array, model) -> jax.Array:
    # return output tensor directly
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch
import torch.nn as nn


# input, model, and output are on the GPU
def solve(input: torch.Tensor, model: nn.Module, output: torch.Tensor):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
from typing import Any, Dict, List

import torch
import torch.nn as nn


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Simple Inference", atol=1e-05, rtol=1e-05, num_gpus=1, access_tier="free"
        )

    def reference_impl(self, input: torch.Tensor, model: nn.Module, output: torch.Tensor):
        assert input.device == output.device
        assert input.dtype == output.dtype

        with torch.no_grad():
            result = model(input)
            output.copy_(result)

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "input": (torch.Tensor, "in"),
            "model": (nn.Module, "in"),
            "output": (torch.Tensor, "out"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        device = "cuda"

        # Create a simple linear model
        model = nn.Linear(2, 2)
        model.weight.data = torch.tensor([[0.5, 1.0], [1.5, 0.5]], dtype=dtype)
        model.bias.data = torch.tensor([0.1, 0.2], dtype=dtype)
        model = model.to(device)

        input = torch.tensor([[1.0, 2.0]], device=device, dtype=dtype)
        output = torch.empty((1, 2), device=device, dtype=dtype)

        return {
            "input": input,
            "model": model,
            "output": output,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        device = "cuda"
        tests = []

        # Test 1: Basic 2->2 linear layer
        model1 = nn.Linear(2, 2)
        model1.weight.data = torch.tensor([[0.5, 1.0], [1.5, 0.5]], dtype=dtype)
        model1.bias.data = torch.tensor([0.1, 0.2], dtype=dtype)
        model1 = model1.to(device)

        tests.append(
            {
                "input": torch.tensor([[1.0, 2.0]], device=device, dtype=dtype),
                "model": model1,
                "output": torch.empty((1, 2), device=device, dtype=dtype),
            }
        )

        # Test 2: Single input/output
        model2 = nn.Linear(1, 1)
        model2.weight.data = torch.tensor([[2.0]], dtype=dtype)
        model2.bias.data = torch.tensor([1.0], dtype=dtype)
        model2 = model2.to(device)

        tests.append(
            {
                "input": torch.tensor([[1.0], [2.0], [3.0]], device=device, dtype=dtype),
                "model": model2,
                "output": torch.empty((3, 1), device=device, dtype=dtype),
            }
        )

        # Test 3: No bias
        model3 = nn.Linear(3, 2, bias=False)
        model3.weight.data = torch.tensor([[1.0, 0.0, -1.0], [0.5, 1.5, 0.0]], dtype=dtype)
        model3 = model3.to(device)

        tests.append(
            {
                "input": torch.tensor(
                    [[1.0, 2.0, 3.0], [0.0, 1.0, -1.0]], device=device, dtype=dtype
                ),
                "model": model3,
                "output": torch.empty((2, 2), device=device, dtype=dtype),
            }
        )

        # Test 4: Batch processing
        model4 = nn.Linear(4, 3)
        model4.weight.data = torch.randn((3, 4), dtype=dtype) * 0.5
        model4.bias.data = torch.randn(3, dtype=dtype) * 0.1
        model4 = model4.to(device)

        tests.append(
            {
                "input": torch.randn((8, 4), device=device, dtype=dtype),
                "model": model4,
                "output": torch.empty((8, 3), device=device, dtype=dtype),
            }
        )

        # Test 5: Larger model
        model5 = nn.Linear(10, 5)
        model5.weight.data = torch.randn((5, 10), dtype=dtype) * 0.3
        model5.bias.data = torch.randn(5, dtype=dtype) * 0.2
        model5 = model5.to(device)

        tests.append(
            {
                "input": torch.randn((16, 10), device=device, dtype=dtype),
                "model": model5,
                "output": torch.empty((16, 5), device=device, dtype=dtype),
            }
        )

        # Test 6: Zero weights
        model6 = nn.Linear(2, 2)
        model6.weight.data = torch.zeros((2, 2), dtype=dtype)
        model6.bias.data = torch.tensor([1.0, -1.0], dtype=dtype)
        model6 = model6.to(device)

        tests.append(
            {
                "input": torch.tensor([[5.0, 10.0]], device=device, dtype=dtype),
                "model": model6,
                "output": torch.empty((1, 2), device=device, dtype=dtype),
            }
        )

        # Test 7: Identity-like transformation
        model7 = nn.Linear(3, 3)
        model7.weight.data = torch.eye(3, dtype=dtype)
        model7.bias.data = torch.zeros(3, dtype=dtype)
        model7 = model7.to(device)

        tests.append(
            {
                "input": torch.tensor(
                    [[1.0, 2.0, 3.0], [-1.0, 0.0, 1.0]], device=device, dtype=dtype
                ),
                "model": model7,
                "output": torch.empty((2, 3), device=device, dtype=dtype),
            }
        )

        # Test 8: Single batch, many features
        model8 = nn.Linear(20, 1)
        model8.weight.data = torch.ones((1, 20), dtype=dtype) * 0.05  # Sum with scaling
        model8.bias.data = torch.tensor([0.0], dtype=dtype)
        model8 = model8.to(device)

        tests.append(
            {
                "input": torch.randn((1, 20), device=device, dtype=dtype),
                "model": model8,
                "output": torch.empty((1, 1), device=device, dtype=dtype),
            }
        )

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        device = "cuda"

        # Large model for performance testing
        model = nn.Linear(512, 256)
        model.weight.data = torch.randn((256, 512), dtype=dtype) * 0.1
        model.bias.data = torch.randn(256, dtype=dtype) * 0.05
        model = model.to(device)

        batch_size = 1000
        input = torch.randn((batch_size, 512), device=device, dtype=dtype)
        output = torch.empty((batch_size, 256), device=device, dtype=dtype)

        return {
            "input": input,
            "model": model,
            "output": output,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
